In [1]:
import plotly.graph_objects as go
import diagnostics
import preprocess
import plots
import utils
import models

### Setup

In [2]:
SNP_file = './data/S&P 500 Index.csv'
VIX_file = './data/S&P 500 VIX.csv'

In [3]:
# Scale is applied to log returns in order to improve numerical stability of models
SCALE = 100

### S&P500 GARCH Log Returns Forecasting

In [4]:
SNP_processed = preprocess.preprocess_data(SNP_file, 'price', 'log_returns')
SNP_processed = utils.upscale_columns(
  SNP_processed, ['log_returns'], SCALE
)
SNP_processed.head()

,price,log_returns
date,,
2015-01-05,2020.58,-1.844722
2015-01-06,2002.61,-0.893327
2015-01-07,2025.90,1.156272
2015-01-08,2062.14,1.773023
2015-01-09,2044.81,-0.843940


In [5]:
snp_garch = models.rolling_garch_price_forecast(SNP_processed, 250, models.Distribution.NORMAL)
snp_garch.head()

,price,log_returns,predicted_price,predicted_log_return,conditional_vol
date,,,,,
2015-01-05,2020.58,-1.844722,NaN,NaN,NaN
2015-01-06,2002.61,-0.893327,NaN,NaN,NaN
2015-01-07,2025.90,1.156272,NaN,NaN,NaN
2015-01-08,2062.14,1.773023,NaN,NaN,NaN
2015-01-09,2044.81,-0.843940,NaN,NaN,NaN


In [6]:
# Rescale back scaled columns
snp_garch = utils.downscale_columns(
  snp_garch, ['conditional_vol', 'log_returns', 'predicted_log_return'], SCALE
)
diagnostics.in_sample_diagnostics(snp_garch['predicted_log_return'], snp_garch['log_returns'], snp_garch['conditional_vol'])

Jarque-Bera test p-value: 0.00000
Ljung-Box (residuals) p-value, 0.70314
Ljung-Box (residuals^2) p-value, 0.00344


In [7]:
snp_garch.head()

,price,log_returns,predicted_price,predicted_log_return,conditional_vol
date,,,,,
2015-01-05,2020.58,-0.018447,NaN,NaN,NaN
2015-01-06,2002.61,-0.008933,NaN,NaN,NaN
2015-01-07,2025.90,0.011563,NaN,NaN,NaN
2015-01-08,2062.14,0.017730,NaN,NaN,NaN
2015-01-09,2044.81,-0.008439,NaN,NaN,NaN


In [8]:
snp_garch_var = snp_garch
snp_garch_var['VaR_99'] = snp_garch_var['predicted_log_return'].rolling(250).quantile(0.01)
diagnostics.compute_var_violations(snp_garch_var, 'VaR_99', 'predicted_log_return')

{'actual_exceedances': 32,
 'expected_exceedances': 20.150000000000016,
 'violation_ratio': 1.5880893300248127}

In [9]:
utils.compute_rmse(snp_garch_var, 'log_returns', 'predicted_log_return')

0.011428973837203748

In [10]:
diagnostics.bernoulli_coverage_test(snp_garch_var, var_col='VaR_99', predicted_col='predicted_log_return')

(0.01453020772441882, 5.972555632361093)

In [11]:
diagnostics.compute_hit_rate(predicted=snp_garch_var['predicted_log_return'], actual=snp_garch_var['log_returns'])


Hit Rate: 44.27%


0.44272076372315033

In [12]:
plots.plot_var_violations(snp_garch_var, var_col='VaR_99', predicted_col='predicted_log_return')

### S&P500 tGARCH Log Returns Forecasting

In [13]:
snp_tgarch = models.rolling_garch_price_forecast(SNP_processed, 250, models.Distribution.T)
snp_tgarch.head()

,price,log_returns,predicted_price,predicted_log_return,conditional_vol
date,,,,,
2015-01-05,2020.58,-1.844722,NaN,NaN,NaN
2015-01-06,2002.61,-0.893327,NaN,NaN,NaN
2015-01-07,2025.90,1.156272,NaN,NaN,NaN
2015-01-08,2062.14,1.773023,NaN,NaN,NaN
2015-01-09,2044.81,-0.843940,NaN,NaN,NaN


In [14]:
# Rescale back scaled columns
snp_tgarch = utils.downscale_columns(
  snp_tgarch, ['conditional_vol', 'log_returns', 'predicted_log_return'], SCALE
)
diagnostics.in_sample_diagnostics(snp_tgarch['predicted_log_return'], snp_tgarch['log_returns'], snp_tgarch['conditional_vol'])

Jarque-Bera test p-value: 0.00000
Ljung-Box (residuals) p-value, 0.81584
Ljung-Box (residuals^2) p-value, 0.25466


In [15]:
snp_tgarch_var = snp_tgarch
snp_tgarch_var['VaR_99'] = snp_tgarch_var['predicted_log_return'].rolling(250).quantile(0.01)
diagnostics.compute_var_violations(snp_tgarch_var, 'VaR_99', 'predicted_log_return')

{'actual_exceedances': 28,
 'expected_exceedances': 20.150000000000016,
 'violation_ratio': 1.389578163771711}

In [16]:
utils.compute_rmse(snp_tgarch_var, 'log_returns', 'predicted_log_return')

0.01143311830627666

In [17]:
diagnostics.bernoulli_coverage_test(snp_tgarch_var, var_col='VaR_99', predicted_col='predicted_log_return')

(0.09695422253005359, 2.7549438133225976)

In [18]:
diagnostics.compute_hit_rate(predicted=snp_tgarch_var['predicted_log_return'], actual=snp_tgarch_var['log_returns'])


Hit Rate: 44.75%


0.44749403341288785

In [19]:
plots.plot_var_violations(snp_tgarch_var, var_col='VaR_99', predicted_col='predicted_log_return')